# 7.1 章节介绍：性能优化基础

第 5、6 章我们完成了「写对」：非量化和全量化 FA 都能跑通、对拍通过。第三部分开始「写快」——逼近硬件理论性能。

## 优化的第一原则：先测量、再建模、后动手

性能优化不是玄学，流程是固定的：

```
测量（拿到真实耗时画像）
  → 建模（Roofline 判断 bound：计算受限 or 带宽受限）
    → 对症下药（不同 bound 用完全不同的药方）
      → 再测量（验证收益，没收益就回滚）
```

**没有建模的优化是盲改**：给带宽受限的算子加计算优化、给计算受限的算子抠搬运，都是南辕北辙。

## 通用三步优化路径

本章给出贯穿后续所有章节的优化步骤（按层次从粗到细）：

| 步骤 | 层次 | 解决的问题 | 本章小节 |
|--|--|--|--|
| ① 均衡性 | 核间 | 别的核算完了在等最慢的核（木桶效应） | [7.4](07.04_balance.ipynb) |
| ② CV 流水、切块 | 引擎间 | Cube 与 Vector 一来一回互相等待；基本块大小决定计算强度 | [7.5](07.05_cv_pipeline.ipynb) |
| ③ 核内流水 | 指令级 | Vector 内部 LD/计算/ST 串行；RegBase 编程把数据驻留寄存器 | [7.6](07.06_inner_pipeline.ipynb) |

三步的顺序就是排查顺序：**先保证每个核的活一样多，再保证两条引擎不互相等，最后抠指令级细节**。反过来做（一上来就抠指令），很可能优化的只是整机耗时的 1%。

## 本章学习路线

| 小节 | 主题 | 回答的问题 |
|--|--|--|
| [7.2 性能建模与 Roofline](07.02_perf_modeling_roofline.ipynb) | W/Q/I、Roofline 公式、混合带宽、基本块计算强度 | 我的算子离上限多远？瓶颈在哪？ |
| [7.3 访存性能](07.03_memory_performance.ipynb) | 带宽账单、UB bank 结构、bank 冲突判定与解法 | 搬运为什么比理论慢？ |
| [7.4 优化步骤①：均衡性](07.04_balance.ipynb) | 分核函数、CoreWeight、贪心分核、有效块统计 | 核间怎么分才不互相等？ |
| [7.5 优化步骤②：CV 流水与切块](07.05_cv_pipeline.ipynb) | 四拍重叠、块大小与计算强度、Softmax VF 的切块 | Cube/Vector 怎么配平？ |
| [7.6 优化步骤③：核内流水](07.06_inner_pipeline.ipynb) | RegBase/VF 编程、vLoop 256B 切分、VF 定位 | 最后一级流水怎么做？ |
| [7.7 章节测验](07.07_chapter_test.ipynb) | 综合测验 | 检验本章掌握程度 |

## 前置知识

- [第5章 非量化 FA 开发](../05_flash_attn_non_quantized/README.md)：有可优化的算子在手（四拍循环、preload、BN1S1）
- [第2章 FlashAttention 原理](../02_fa_principle/README.md)中的存储层次
- [第1章 昇腾开发基础](../01_ascend_dev_basics/README.md)：MTE2/MTE1/MTE3、Cube/Vector/Scalar 分工